# Radiologist Annotation Validation

Evaluating radiologist judgments on cases where CT-RATE and GPT-5.4 labels disagreed.

| Set | Cases | Target labels |
|---|---|---|
| General discordance set | 100 | 16 labels with κ > 0.80 |
| Focus discordance set | 100 | Lymphadenopathy (κ = 0.310) |

**Judgment categories**
- `ct_rate`: CT-RATE label is more consistent with the report text
- `gpt`: GPT-5.4 label is more consistent with the report text
- `both`: Both labels are acceptable
- `neither`: Neither label is sufficiently consistent with the report text

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from scipy import stats

FIGURES_DIR = Path("../figures")
TABLES_DIR  = Path("../tables")

plt.rcParams.update({
    "figure.dpi": 150,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

COLORS = {
    "gpt":     "#DD8452",
    "ct_rate": "#4C72B0",
    "both":    "#8172B2",
    "neither": "#C44E52",
}
LABELS = {
    "gpt":     "GPT-5.4",
    "ct_rate": "CT-RATE",
    "both":    "Both acceptable",
    "neither": "Neither consistent",
}

In [ ]:
general = pd.read_csv("../data/general_annotations_v2.csv")
# display(general)
general = general[general["annotator"]=="高松篤"].reset_index(drop=True)
focus   = pd.read_csv("../data/focus_annotations_v2.csv")
focus = focus[focus["annotator"]=="高松篤"].reset_index(drop=True)

print(f"General discordance set: {len(general)} cases")
print(general["judgment"].value_counts().rename(LABELS).to_string())
print()
print(f"Focus discordance set (Lymphadenopathy): {len(focus)} cases")
print(focus["judgment"].value_counts().rename(LABELS).to_string())

In [ ]:
def wilson_ci(k, n, alpha=0.05):
    """Wilson score 95% CI for a proportion."""
    if n == 0:
        return np.nan, np.nan
    z = stats.norm.ppf(1 - alpha / 2)
    p = k / n
    denom  = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return center - margin, center + margin


def summarize(df, label=""):
    """Compute judgment counts, GPT% with 95% CI, and binomial test p-value."""
    vc       = df["judgment"].value_counts()
    n_gpt    = int(vc.get("gpt", 0))
    n_ct     = int(vc.get("ct_rate", 0))
    n_both   = int(vc.get("both", 0))
    n_neit   = int(vc.get("neither", 0))
    n        = len(df)
    n_dec    = n_gpt + n_ct  # decisive cases only

    if n_dec > 0:
        p_gpt        = n_gpt / n_dec
        ci_lo, ci_hi = wilson_ci(n_gpt, n_dec)
        pval         = stats.binomtest(n_gpt, n_dec, p=0.5, alternative="two-sided").pvalue
    else:
        p_gpt = ci_lo = ci_hi = pval = np.nan

    return dict(
        group=label, n=n,
        gpt_correct=n_gpt, ct_rate_correct=n_ct, both=n_both, neither=n_neit,
        n_decisive=n_dec,
        gpt_correct_pct=p_gpt * 100 if not np.isnan(p_gpt) else np.nan,
        CI_lo=ci_lo * 100, CI_hi=ci_hi * 100,
        p_value=pval,
    )

---
## 1. General Discordance Label Set

Cases discordant on 16 labels with κ > 0.80 (CT+/GPT− 50 cases, CT−/GPT+ 50 cases).

In [ ]:
# ── stats by direction ────────────────────────────────────────────────────────
gen_rows = [summarize(general, "Overall")]
for direction, grp in general.groupby("direction"):
    dlabel = "CT+/GPT− (CT-RATE positive / GPT-5.4 negative)" if direction == "ct_pos_gpt_neg" \
             else "CT−/GPT+ (CT-RATE negative / GPT-5.4 positive)"
    gen_rows.append(summarize(grp, dlabel))

gen_stats = pd.DataFrame(gen_rows)

# Display table
disp = gen_stats.copy()
disp["gpt_correct_pct (95%CI)"] = disp.apply(
    lambda r: f"{r['gpt_correct_pct']:.1f}% ({r['CI_lo']:.1f}–{r['CI_hi']:.1f})", axis=1
)
disp["p_value"] = disp["p_value"].apply(lambda p: f"{p:.4f}" if p >= 0.001 else "<0.001")

display(disp[["group", "n", "gpt_correct", "ct_rate_correct", "both", "neither",
              "gpt_correct_pct (95%CI)", "p_value"]].set_index("group"))

gen_stats.to_csv(TABLES_DIR / "validation_general_stats.csv", index=False)

In [ ]:
# ── plot: judgment distribution by direction ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
titles = ["Overall", "CT+/GPT− direction", "CT−/GPT+ direction"]
subsets = [general,
           general[general["direction"] == "ct_pos_gpt_neg"],
           general[general["direction"] == "ct_neg_gpt_pos"]]

for ax, title, df in zip(axes, titles, subsets):
    vc = df["judgment"].value_counts()
    cats = [c for c in ["gpt", "ct_rate", "both", "neither"] if c in vc.index]
    vals = [vc.get(c, 0) for c in cats]
    bars = ax.bar([LABELS[c] for c in cats], vals,
                  color=[COLORS[c] for c in cats], width=0.6)
    ax.bar_label(bars, padding=2, fontsize=9)
    ax.set_title(f"General discordance set\n{title} (n={len(df)})", fontsize=10)
    ax.set_ylim(0, max(vals) * 1.25)
    ax.set_ylabel("Count")
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig(FIGURES_DIR / "validation_general_direction.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── per-label breakdown ───────────────────────────────────────────────────────
label_rows = []
for lname, grp in general.groupby("label_name"):
    label_rows.append(summarize(grp, lname))

label_stats = pd.DataFrame(label_rows).sort_values("gpt_correct_pct", ascending=False)

disp_l = label_stats.copy()
disp_l["gpt_correct_pct (95%CI)"] = disp_l.apply(
    lambda r: f"{r['gpt_correct_pct']:.1f}% ({r['CI_lo']:.1f}–{r['CI_hi']:.1f})"
              if not np.isnan(r['gpt_correct_pct']) else "—", axis=1
)
disp_l["p_value"] = disp_l["p_value"].apply(
    lambda p: f"{p:.3f}" if (not np.isnan(p) and p >= 0.001) else ("<0.001" if not np.isnan(p) else "—")
)
display(disp_l[["group", "n", "gpt_correct", "ct_rate_correct", "both",
                "gpt_correct_pct (95%CI)", "p_value"]].set_index("group"))

label_stats.to_csv(TABLES_DIR / "validation_general_per_label.csv", index=False)

In [ ]:
# # ── per-label diverging bar ────────────────────────────────────────────────────
# ls = label_stats.set_index("group").sort_values("gpt_correct_pct", ascending=False)

# label_names = ls.index.tolist()
# n      = len(label_names)
# y      = np.arange(n)
# g_vals = ls["gpt_correct"].values.astype(float)
# c_vals = ls["ct_rate_correct"].values.astype(float)
# x_max  = int(max(g_vals.max(), c_vals.max())) + 2

# fig, ax = plt.subplots(figsize=(9, 5))

# ax.barh(y,  g_vals, height=0.5, color=COLORS["gpt"],     label=LABELS["gpt"])
# ax.barh(y, -c_vals, height=0.5, color=COLORS["ct_rate"], label=LABELS["ct_rate"])

# # Count labels — non-zero only to avoid clutter
# for i, (g, c) in enumerate(zip(g_vals, c_vals)):
#     if g > 0:
#         ax.text(g + 0.2, i, str(int(g)), va="center", ha="left",  fontsize=8)
#     if c > 0:
#         ax.text(-c - 0.2, i, str(int(c)), va="center", ha="right", fontsize=8)

# ax.axvline(0, color="black", linewidth=0.8)
# ax.set_yticks(y)
# ax.set_yticklabels(label_names, fontsize=9)
# ax.set_xlim(-x_max, x_max)
# ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: str(int(abs(x)))))
# ax.set_xlabel("← CT-RATE correct   |   GPT-5.4 correct →")
# ax.set_title("General discordance set: GPT-5.4 vs CT-RATE correct per label", fontsize=11)
# ax.legend(fontsize=9, loc="lower left")

# plt.tight_layout()
# plt.savefig(FIGURES_DIR / "validation_general_per_label.png", bbox_inches="tight")
# plt.show()

In [ ]:
# ── per-label diverging bar ────────────────────────────────────────────────────
# ls = label_stats.set_index("group").sort_values("gpt_correct_pct", ascending=False)
ls = label_stats.set_index("group")
ls["_ct_zero"] = (ls["ct_rate_correct"] == 0).astype(int)
ls = ls.sort_values(
    ["_ct_zero", "ct_rate_correct", "gpt_correct"],
    ascending=[False, True, False]
)

label_names = ls.index.tolist()
n      = len(label_names)
y      = np.arange(n)
g_vals = ls["gpt_correct"].values.astype(float)
c_vals = ls["ct_rate_correct"].values.astype(float)
x_max  = int(max(g_vals.max(), c_vals.max())) + 2

fig, ax = plt.subplots(figsize=(9, 5))

ax.barh(y,  g_vals, height=0.5, color=COLORS["gpt"],     label=LABELS["gpt"])
ax.barh(y, -c_vals, height=0.5, color=COLORS["ct_rate"], label=LABELS["ct_rate"])

# Count labels — non-zero only to avoid clutter
for i, (g, c) in enumerate(zip(g_vals, c_vals)):
    if g > 0:
        ax.text(g + 0.2, i, str(int(g)), va="center", ha="left",  fontsize=8)
    if c > 0:
        ax.text(-c - 0.2, i, str(int(c)), va="center", ha="right", fontsize=8)

ax.axvline(0, color="black", linewidth=0.8)
ax.set_yticks(y)
ax.set_yticklabels(label_names, fontsize=9)
ax.set_xlim(-x_max, x_max)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: str(int(abs(x)))))
ax.set_xlabel("← CT-RATE correct   |   GPT-5.4 correct →")
ax.set_title("General discordance set: GPT-5.4 vs CT-RATE correct per label", fontsize=11)
ax.legend(fontsize=9, loc="lower left")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "validation_general_per_label.png", bbox_inches="tight")
plt.show()

---
## 2. Focus Discordance Label Set (Lymphadenopathy)

The label with the lowest Cohen's κ of **0.310** between CT-RATE and GPT-5.4.  
Evaluating discordant cases (CT+/GPT− 50 cases, CT−/GPT+ 50 cases).

In [ ]:
# ── stats by direction ────────────────────────────────────────────────────────
foc_rows = [summarize(focus, "Overall (Lymphadenopathy)")]
for direction, grp in focus.groupby("direction"):
    dlabel = "CT+/GPT− (CT-RATE positive / GPT-5.4 negative)" if direction == "ct_pos_gpt_neg" \
             else "CT−/GPT+ (CT-RATE negative / GPT-5.4 positive)"
    foc_rows.append(summarize(grp, dlabel))

foc_stats = pd.DataFrame(foc_rows)

disp_f = foc_stats.copy()
disp_f["gpt_correct_pct (95%CI)"] = disp_f.apply(
    lambda r: f"{r['gpt_correct_pct']:.1f}% ({r['CI_lo']:.1f}–{r['CI_hi']:.1f})", axis=1
)
disp_f["p_value"] = disp_f["p_value"].apply(lambda p: f"{p:.4f}" if p >= 0.001 else "<0.001")

display(disp_f[["group", "n", "gpt_correct", "ct_rate_correct", "both", "neither",
                "gpt_correct_pct (95%CI)", "p_value"]].set_index("group"))

foc_stats.to_csv(TABLES_DIR / "validation_focus_stats.csv", index=False)

In [ ]:
# ── plot: judgment distribution by direction ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
titles  = ["Overall",
           "CT+/GPT− direction\n(possible CT-RATE over-detection)",
           "CT−/GPT+ direction\n(possible CT-RATE under-detection)"]
subsets = [focus,
           focus[focus["direction"] == "ct_pos_gpt_neg"],
           focus[focus["direction"] == "ct_neg_gpt_pos"]]

for ax, title, df in zip(axes, titles, subsets):
    vc   = df["judgment"].value_counts()
    cats = [c for c in ["gpt", "ct_rate", "both", "neither"] if c in vc.index]
    vals = [vc.get(c, 0) for c in cats]
    bars = ax.bar([LABELS[c] for c in cats], vals,
                  color=[COLORS[c] for c in cats], width=0.6)
    ax.bar_label(bars, padding=2, fontsize=9)
    ax.set_title(f"Focus discordance set (Lymphadenopathy)\n{title} (n={len(df)})", fontsize=9)
    ax.set_ylim(0, max(vals) * 1.25)
    ax.set_ylabel("Count")
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig(FIGURES_DIR / "validation_focus_direction.png", bbox_inches="tight")
plt.show()

---
## 3. Overall Summary

In [ ]:
# ── combined summary table ────────────────────────────────────────────────────
summary_rows = []

# General: overall + by direction
summary_rows.append({"set": "General discordance", **summarize(general, "Overall")})
for d, g in general.groupby("direction"):
    dlabel = "CT+/GPT−" if d == "ct_pos_gpt_neg" else "CT−/GPT+"
    summary_rows.append({"set": "General discordance", **summarize(g, dlabel)})

# Focus: overall + by direction
summary_rows.append({"set": "Focus discordance\n(Lymphadenopathy)", **summarize(focus, "Overall")})
for d, g in focus.groupby("direction"):
    dlabel = "CT+/GPT−" if d == "ct_pos_gpt_neg" else "CT−/GPT+"
    summary_rows.append({"set": "Focus discordance\n(Lymphadenopathy)", **summarize(g, dlabel)})

summary = pd.DataFrame(summary_rows)
summary["gpt_correct_pct (95%CI)"] = summary.apply(
    lambda r: f"{r['gpt_correct_pct']:.1f}% ({r['CI_lo']:.1f}–{r['CI_hi']:.1f})", axis=1
)
summary["p_value"] = summary["p_value"].apply(lambda p: f"{p:.4f}" if p >= 0.001 else "<0.001")

display(summary[["set", "group", "n", "gpt_correct", "ct_rate_correct", "both",
                 "gpt_correct_pct (95%CI)", "p_value"]].set_index(["set", "group"]))

summary.to_csv(TABLES_DIR / "validation_summary.csv", index=False)
summary.to_excel(TABLES_DIR / "validation_summary.xlsx", index=False)

In [ ]:
# ── forest-plot style summary ─────────────────────────────────────────────────
plot_rows = [
    ("General\nOverall",       summarize(general)),
    ("General\nCT+/GPT−",     summarize(general[general["direction"]=="ct_pos_gpt_neg"])),
    ("General\nCT−/GPT+",     summarize(general[general["direction"]=="ct_neg_gpt_pos"])),
    ("Focus (LAP)\nOverall",   summarize(focus)),
    ("Focus (LAP)\nCT+/GPT−", summarize(focus[focus["direction"]=="ct_pos_gpt_neg"])),
    ("Focus (LAP)\nCT−/GPT+", summarize(focus[focus["direction"]=="ct_neg_gpt_pos"])),
]

ylabels = [r[0] for r in plot_rows]
centers = [r[1]["gpt_correct_pct"] / 100 for r in plot_rows]
lo      = [r[1]["CI_lo"] / 100           for r in plot_rows]
hi      = [r[1]["CI_hi"] / 100           for r in plot_rows]
errs    = [[c - l for c, l in zip(centers, lo)],
           [h - c for h, c in zip(hi, centers)]]

fig, ax = plt.subplots(figsize=(7, 5))

colors = ["#DD8452" if c > 0.5 else "#4C72B0" for c in centers]
y_pos  = range(len(ylabels))

ax.barh(y_pos, centers, xerr=errs, color=colors, height=0.5,
        align="center", capsize=4, error_kw={"linewidth": 1.5})
ax.axvline(0.5, color="gray", linestyle="--", linewidth=1, label="GPT = CT-RATE (50%)")
ax.set_yticks(list(y_pos))
ax.set_yticklabels(ylabels, fontsize=9)
ax.set_xlabel("GPT-5.4 correct rate (95% Wilson CI)")
ax.set_title("GPT-5.4 correct rate in discordant cases", fontsize=11)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x*100:.0f}%"))
ax.set_xlim(0, 1)
ax.invert_yaxis()

patch_gpt = mpatches.Patch(color="#DD8452", label="GPT-5.4 superior")
patch_ct  = mpatches.Patch(color="#4C72B0", label="CT-RATE superior")
ax.legend(handles=[patch_gpt, patch_ct,
                   plt.Line2D([0], [0], color="gray", linestyle="--", label="Reference line (50%)")],
          fontsize=8, loc="lower right")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "validation_summary_forest.png", bbox_inches="tight")
plt.show()

---
## 4. Comment Review of CT-RATE Correct Cases

Reviewing comments for cases where CT-RATE label was judged correct.

In [ ]:
# ── general: CT-RATE correct cases ───────────────────────────────────────────
gen_ct = general[general["judgment"] == "ct_rate"][["volume_name", "label_name",
                                                     "direction", "comment"]].copy()
gen_ct.index = range(1, len(gen_ct) + 1)
print(f"General discordance set: CT-RATE correct {len(gen_ct)} cases\n")
display(gen_ct)

In [ ]:
# ── focus: CT-RATE correct cases ─────────────────────────────────────────────
foc_ct = focus[focus["judgment"] == "ct_rate"][["volume_name", "label_name",
                                                  "direction", "comment"]].copy()
foc_ct.index = range(1, len(foc_ct) + 1)
print(f"Focus discordance set (Lymphadenopathy): CT-RATE correct {len(foc_ct)} cases\n")
display(foc_ct)

In [ ]:
# ── both/neither cases ────────────────────────────────────────────────────────
ambiguous = pd.concat([
    general[general["judgment"].isin(["both", "neither"])].assign(set="General discordance"),
    focus[focus["judgment"].isin(["both", "neither"])].assign(set="Focus discordance"),
])[["set", "volume_name", "label_name", "judgment", "direction", "comment"]]

if len(ambiguous):
    print(f"Both acceptable / Neither consistent: {len(ambiguous)} cases")
    display(ambiguous.reset_index(drop=True))
else:
    print("No cases of 'both acceptable' or 'neither consistent'.")